# Structured Outputs & Practical Applications

This notebook covers:

1. **Structured Output Fundamentals** — Getting typed, validated data from LLMs
2. **Practical Application** — Building a Resume Analyzer & Job-Fit Scorer

**Docs**: [Structured Output](https://docs.langchain.com/oss/python/langchain/structured-output)

## Setup

In [1]:
%pip install -qU langchain langchain-openai langgraph python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"
print("Environment ready!")

Environment ready!


---
## 1. Structured Output Fundamentals

Instead of getting free-form text from an LLM, **structured output** returns typed Python objects.

Use the `response_format` parameter on `create_agent()`.

### Why it matters:
- Parse-free: no regex or JSON parsing needed
- Type-safe: Pydantic validates the output
- Composable: feed structured results directly into downstream code

### Approach 1: TypedDict (simplest, returns dict)

In [3]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    name: str
    email: str
    phone: str

agent = create_agent(
    model="openai:gpt-4o-mini",
    response_format=ContactInfo,
)

result = agent.invoke({
    "messages": "Extract contact info: Hi I'm Jane Smith, reach me at jane@example.com or (555) 123-4567"
})

contact = result["structured_response"]
print(contact)
print(type(contact))  # dict

{'name': 'Jane Smith', 'email': 'jane@example.com', 'phone': '(555) 123-4567'}
<class 'dict'>


### Approach 2: Pydantic (recommended — validation + descriptions)

In [4]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal

class MovieReview(BaseModel):
    """Structured analysis of a movie review."""
    title: str = Field(description="Movie title mentioned in the review")
    sentiment: Literal["positive", "negative", "mixed"] = Field(description="Overall sentiment")
    rating: int = Field(description="Rating from 1-10", ge=1, le=10)
    key_points: List[str] = Field(description="Main points from the review")
    recommended: bool = Field(description="Whether the reviewer recommends it")

review_agent = create_agent(
    model="openai:gpt-5.4-mini",
    response_format=MovieReview,
)

result = review_agent.invoke({
    "messages": """Analyze this review:
    'Inception is a masterpiece. The layered dream sequences are mind-bending,
    the acting is superb, and Hans Zimmer's score is unforgettable. 
    My only gripe is the pacing in the first 20 minutes. Must watch!'"""
})

review = result["structured_response"]
print(f"Title: {review.title}")
print(f"Sentiment: {review.sentiment}")
print(f"Rating: {review.rating}/10")
print(f"Key Points: {review.key_points}")
print(f"Recommended: {review.recommended}")

Title: Inception
Sentiment: positive
Rating: 9/10
Key Points: ['Described as a masterpiece', 'Layered dream sequences are mind-bending', 'Acting is superb', "Hans Zimmer's score is unforgettable", 'Minor gripe about pacing in the first 20 minutes', 'Strong recommendation: must watch']
Recommended: True


### Strategies: ProviderStrategy vs ToolStrategy

| Strategy | How it works | Best for |
|----------|-------------|----------|
| `ProviderStrategy` | Uses the model's native structured output API | OpenAI, Anthropic, Gemini |
| `ToolStrategy` | Uses tool calling to extract structured data | Any model with tool support |
| Auto (pass schema directly) | LangChain picks the best strategy | Getting started |

When you pass a schema directly to `response_format`, LangChain auto-selects the best strategy.

In [5]:
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy

# Explicit ProviderStrategy (native API — fastest)
agent_provider = create_agent(
    model="openai:gpt-5.4-mini",
    response_format=ProviderStrategy(schema=MovieReview),
)

# Explicit ToolStrategy (universal fallback — works with tools too)
agent_tool = create_agent(
    model="openai:gpt-5.4-mini",
    response_format=ToolStrategy(schema=MovieReview),
)

# Both produce the same structured output
r1 = agent_provider.invoke({"messages": "Review: The Matrix is amazing, 9/10, highly recommend!"})
r2 = agent_tool.invoke({"messages": "Review: The Matrix is amazing, 9/10, highly recommend!"})

print(f"Provider: {r1['structured_response'].title} — {r1['structured_response'].sentiment}")
print(f"Tool:     {r2['structured_response'].title} — {r2['structured_response'].sentiment}")

Provider: The Matrix — positive
Tool:     The Matrix — positive


---
## 2. Practical Application: Resume Analyzer & Job-Fit Scorer

Let's build something useful — an agent that:
1. Takes a **job description** and a **resume**
2. Extracts structured candidate profile data
3. Produces a **scored evaluation** with skills match, gaps, and a recommendation

This demonstrates **nested Pydantic models**, `Literal` enums, constrained fields, and real-world utility.

### Step 1: Define the Data Models

In [6]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal


class SkillAssessment(BaseModel):
    """Assessment of a single skill."""
    skill_name: str = Field(description="Name of the skill")
    proficiency: Literal["expert", "proficient", "familiar", "none"] = Field(
        description="Candidate's proficiency level based on resume evidence"
    )
    evidence: str = Field(description="Brief evidence from the resume supporting this assessment")


class CandidateProfile(BaseModel):
    """Extracted candidate information."""
    name: str = Field(description="Candidate's full name")
    years_experience: int = Field(description="Total years of professional experience", ge=0)
    current_role: str = Field(description="Current or most recent job title")
    education: str = Field(description="Highest education level and field")
    top_skills: List[str] = Field(description="Top 5 technical skills from the resume")


class JobFitEvaluation(BaseModel):
    """Complete job-fit evaluation of a candidate."""
    candidate: CandidateProfile = Field(description="Extracted candidate profile")
    skill_assessments: List[SkillAssessment] = Field(
        description="Assessment of each required skill from the job description"
    )
    fit_score: int = Field(
        description="Overall fit score from 0-100", ge=0, le=100
    )
    strengths: List[str] = Field(description="Key strengths relative to the role")
    gaps: List[str] = Field(description="Missing skills or experience gaps")
    recommendation: Literal["strong_hire", "hire", "maybe", "no_hire"] = Field(
        description="Hiring recommendation"
    )
    summary: str = Field(description="2-3 sentence overall assessment")

print("Models defined! Schema has", len(JobFitEvaluation.model_fields), "top-level fields")
print("Nested models: CandidateProfile, SkillAssessment")

Models defined! Schema has 7 top-level fields
Nested models: CandidateProfile, SkillAssessment


### Step 2: Why Structured Output Matters — The Contrast

First, let's see what happens **without** structured output:

In [8]:
SAMPLE_JOB = """
Senior Python Developer — AI/ML Team

Requirements:
- 5+ years Python experience
- Experience with LangChain, LLM frameworks
- Strong SQL and database skills
- FastAPI or Django for API development
- Cloud experience (AWS or GCP)
- Nice to have: Kubernetes, MLOps
"""

SAMPLE_RESUME = """
Maria Santos
Senior Software Engineer | 7 years experience

EXPERIENCE:
- Senior Software Engineer at TechCorp (2022-present)
  Built ML pipelines using Python, LangChain, and FastAPI.
  Deployed models on AWS (ECS, Lambda, S3).
  Led a team of 3 engineers on the RAG search platform.

- Software Engineer at DataInc (2019-2022)
  Developed REST APIs with Django and PostgreSQL.
  Wrote complex SQL queries for analytics dashboard.
  Implemented CI/CD pipelines with GitHub Actions.

- Junior Developer at StartupXYZ (2017-2019)
  Full-stack development with Python and React.

EDUCATION:
- M.S. Computer Science, University of Lisbon (2017)

SKILLS: Python, LangChain, FastAPI, Django, PostgreSQL, AWS, Docker, Git, React
"""

# Unstructured approach — just text
unstructured_agent = create_agent(model="openai:gpt-5.4-mini", tools=[])

result = unstructured_agent.invoke({
    "messages": f"Evaluate this candidate for the role:\n\nJOB:\n{SAMPLE_JOB}\n\nRESUME:\n{SAMPLE_RESUME}"
})
print("--- Unstructured output (just text) ---")
print(result["messages"][-1].content[:500], "...")

--- Unstructured output (just text) ---
**Overall assessment: Strong fit**

Maria looks like a very good match for this **Senior Python Developer — AI/ML Team** role.

### Requirement match
- **5+ years Python experience:** Yes — 7 years total, with Python across multiple roles.
- **LangChain / LLM frameworks:** Yes — direct experience with **LangChain** and a **RAG search platform**.
- **Strong SQL and database skills:** Yes — **PostgreSQL** and complex SQL queries mentioned.
- **FastAPI or Django for API development:** Yes — both ** ...


The text above is useful to read, but you can't programmatically access the score, recommendation, or skill gaps. Let's fix that.

### Step 3: Build the Structured Agent

In [10]:
resume_agent = create_agent(
    model="openai:gpt-5.4-mini",
    system_prompt="""You are an expert technical recruiter. Evaluate candidates against job descriptions.
    
    Be objective and evidence-based:
    - Only assess skills explicitly mentioned or clearly implied in the resume
    - Set proficiency to 'none' for skills not evidenced in the resume
    - Score fit_score based on coverage of required skills (not nice-to-haves)
    - Be specific in your evidence citations""",
    response_format=JobFitEvaluation,
)

result = resume_agent.invoke({
    "messages": f"Evaluate this candidate:\n\nJOB DESCRIPTION:\n{SAMPLE_JOB}\n\nRESUME:\n{SAMPLE_RESUME}"
})

evaluation = result["structured_response"]
print(type(evaluation))  # JobFitEvaluation

<class '__main__.JobFitEvaluation'>


### Step 4: Access the Structured Data

In [11]:
# Access nested fields programmatically
print(f"Candidate: {evaluation.candidate.name}")
print(f"Experience: {evaluation.candidate.years_experience} years")
print(f"Current Role: {evaluation.candidate.current_role}")
print(f"Education: {evaluation.candidate.education}")
print(f"Top Skills: {', '.join(evaluation.candidate.top_skills)}")
print()
print(f"FIT SCORE: {evaluation.fit_score}/100")
print(f"RECOMMENDATION: {evaluation.recommendation.upper()}")
print()
print(f"Summary: {evaluation.summary}")

Candidate: Maria Santos
Experience: 7 years
Current Role: Senior Software Engineer
Education: M.S. Computer Science, University of Lisbon
Top Skills: Python, LangChain, FastAPI, Django, AWS

FIT SCORE: 93/100
RECOMMENDATION: STRONG_HIRE

Summary: Maria Santos is an excellent match for this Senior Python Developer role, with direct evidence of Python, LangChain, FastAPI/Django, SQL, and AWS experience. Her background building ML pipelines and a RAG search platform is especially relevant to an AI/ML team, and she clearly exceeds the Python experience requirement.


In [12]:
# Skill-by-skill breakdown
print("\nSkill Assessments:")
print("-" * 60)
for skill in evaluation.skill_assessments:
    icon = {"expert": "★★★", "proficient": "★★☆", "familiar": "★☆☆", "none": "☆☆☆"}[skill.proficiency]
    print(f"  {icon} {skill.skill_name:20s} [{skill.proficiency}]")
    print(f"      Evidence: {skill.evidence}")

print("\nStrengths:", evaluation.strengths)
print("Gaps:", evaluation.gaps)


Skill Assessments:
------------------------------------------------------------
  ★★★ Python experience (5+ years) [expert]
      Evidence: 7 years experience total; roles from 2017-present and repeated Python use in ML pipelines and full-stack development.
  ★★☆ LangChain / LLM frameworks [proficient]
      Evidence: Built ML pipelines using Python, LangChain, and FastAPI; led a RAG search platform.
  ★★☆ SQL and database skills [proficient]
      Evidence: Developed REST APIs with Django and PostgreSQL; wrote complex SQL queries for analytics dashboard.
  ★★★ FastAPI or Django for API development [expert]
      Evidence: Built ML pipelines using Python, LangChain, and FastAPI; previously developed REST APIs with Django.
  ★★☆ Cloud experience (AWS or GCP) [proficient]
      Evidence: Deployed models on AWS using ECS, Lambda, and S3.
  ☆☆☆ Kubernetes           [none]
      Evidence: No Kubernetes experience mentioned on the resume.
  ★☆☆ MLOps                [familiar]
      Evidence

### Step 5: Batch Evaluation — Compare Multiple Candidates

In [13]:
RESUME_2 = """
James Chen
ML Engineer | 3 years experience

EXPERIENCE:
- ML Engineer at AIStartup (2023-present)
  Fine-tuned LLMs using PyTorch and Hugging Face.
  Built evaluation pipelines for model performance.
  Some experience with LangChain for prototyping.

- Data Analyst at BigCo (2021-2023)
  Python scripting for data analysis.
  Basic SQL queries and Tableau dashboards.

EDUCATION:
- B.S. Statistics, UCLA (2021)

SKILLS: Python, PyTorch, Hugging Face, LangChain (basic), SQL, Tableau
"""

RESUME_3 = """
Aisha Patel
DevOps & Platform Engineer | 8 years experience

EXPERIENCE:
- Platform Lead at CloudNative Inc (2020-present)
  Designed Kubernetes clusters on GCP for ML workloads.
  Built MLOps pipelines with Kubeflow and Airflow.
  Python automation scripts for infrastructure.

- DevOps Engineer at ScaleUp (2018-2020)
  AWS infrastructure (EC2, RDS, EKS).
  CI/CD with Jenkins and Terraform.

- Systems Admin at LocalTech (2016-2018)
  Linux administration and shell scripting.

EDUCATION:
- B.S. Information Systems, Georgia Tech (2016)

SKILLS: Kubernetes, GCP, AWS, Python, Terraform, Docker, Airflow, MLOps, Linux
"""

# Evaluate all candidates
candidates = [
    ("Maria Santos", SAMPLE_RESUME),
    ("James Chen", RESUME_2),
    ("Aisha Patel", RESUME_3),
]

evaluations = []
for name, resume in candidates:
    r = resume_agent.invoke({
        "messages": f"Evaluate:\n\nJOB:\n{SAMPLE_JOB}\n\nRESUME:\n{resume}"
    })
    evaluations.append(r["structured_response"])
    print(f"Evaluated: {name}")

Evaluated: Maria Santos
Evaluated: James Chen
Evaluated: Aisha Patel


In [14]:
# Compare candidates — this is only possible with structured output!
print("\n" + "=" * 60)
print("CANDIDATE COMPARISON")
print("=" * 60)
print(f"{'Candidate':20s} {'Score':>6s}   {'Recommendation':15s}   Key Gap")
print("-" * 60)

# Sort by score
for ev in sorted(evaluations, key=lambda x: x.fit_score, reverse=True):
    gap = ev.gaps[0] if ev.gaps else "None"
    print(f"{ev.candidate.name:20s} {ev.fit_score:>5d}%   {ev.recommendation:15s}   {gap}")

print("\nTop candidate:", max(evaluations, key=lambda x: x.fit_score).candidate.name)


CANDIDATE COMPARISON
Candidate             Score   Recommendation    Key Gap
------------------------------------------------------------
Maria Santos            96%   strong_hire       No explicit Kubernetes experience.
Aisha Patel             48%   maybe             No evidence of LangChain or other LLM framework experience
James Chen              38%   no_hire           Does not meet the 5+ years Python requirement.

Top candidate: Maria Santos


### Try It Yourself!

Paste your own resume text below and evaluate it against any job description.

In [ ]:
# YOUR_JOB = """Paste a job description here"""
# YOUR_RESUME = """Paste your resume text here"""

# result = resume_agent.invoke({
#     "messages": f"Evaluate:\n\nJOB:\n{YOUR_JOB}\n\nRESUME:\n{YOUR_RESUME}"
# })
# ev = result["structured_response"]
# print(f"Score: {ev.fit_score}/100 — {ev.recommendation}")
# print(f"Summary: {ev.summary}")

---
## Summary

| Concept | API | When to Use |
|---------|-----|-------------|
| **Auto-selection** | `response_format=MyModel` | Getting started, simple cases |
| **ProviderStrategy** | `ProviderStrategy(schema=MyModel)` | Best performance, native support |
| **ToolStrategy** | `ToolStrategy(schema=MyModel)` | When you need tools + structured output |
| **TypedDict** | `class X(TypedDict)` | Quick prototyping (returns dict) |
| **Pydantic** | `class X(BaseModel)` | Production (validation + descriptions) |

**Key takeaway**: Structured output transforms LLMs from text generators into **data extraction engines**.

**Next**: Check out the [demo/](../demo/) directory for a deployable Chat-Over-Docs application.